In [3]:
"""

This module computes and analyzes comprehensive evaluation metrics for outlier 
detection algorithms across multiple converted datasets and conversion methods.

Key Features:
  - Calculation of ranking-based metrics (AUC, P@n, AP, Max-F1)
  - Adjustment for class imbalance using random baselines
  - Batch processing of multiple dataset groups
  - Statistical aggregation and export
  - Dataset descriptor generation

Organization:
  1. Imports & Dependencies
  2. Configuration & Constants
  3. Basic Ranking-Based Metrics
  4. Adjusted Metrics (Normalized by Class Imbalance)
  5. Aggregation & Analysis Functions
  6. Main Execution Pipeline
"""

import os
import re
import math
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
import warnings
from tqdm import tqdm
from typing import Dict, List, Tuple

# Suppress all library warnings for cleaner output
warnings.filterwarnings("ignore")

In [5]:

# =============================================================================
# SECTION 1: Configuration & Constants
# =============================================================================

# Datasets to exclude from analysis (can be extended with specific dataset names)
EXCLUDED_DATASETS = [
    # Add dataset names to exclude here
]

# Conversion methods to process
CONVERSION_METHODS_BINARY = ['BIN', 'BINDOWN']
CONVERSION_METHODS_NON_BINARY = ['EXC', 'EXCDOWN', 'GRO', 'GRODOWN']
CONVERSION_METHODS_ALL = CONVERSION_METHODS_BINARY + CONVERSION_METHODS_NON_BINARY

# Metrics column names
METRICS_COLUMNS = ['algorithm', 'dataset', 'method', 'AUC', 'P@n', 'AP', 'Max-F1']

# Statistical aggregation functions
AGGREGATION_FUNCTIONS = {
    'AUC': ['mean', 'std'],
    'P@n': ['mean', 'std'],
    'AP': ['mean', 'std'],
    'Max-F1': ['mean', 'std'],
}


# =============================================================================
# SECTION 2: Basic Ranking-Based Metrics
# =============================================================================

def prepare_binary_predictions(correctness: pd.Series, instance_type: pd.Series,
                               scope: str = None) -> Tuple[list, list]:
    """
    Convert correctness and type columns to binary labels.
    
    Transforms boolean correctness values and type strings into binary
    classification targets suitable for sklearn metrics.
    
    Binary encoding:
    - correctness: True → 1 (correct), False → 0 (incorrect)
    - type: 'I' → 1 (inlier), 'O' → 0 (outlier) when scope='I'
    - type: 'O' → 1 (outlier), 'I' → 0 (inlier) when scope='O'
    
    Args:
        correctness: Series of boolean correctness values (True/False)
        instance_type: Series of type strings ('I' for inlier, 'O' for outlier)
        scope: Filter scope:
               - None or 'I': y_true=1 for inliers
               - 'O': y_true=1 for outliers
        
    Returns:
        Tuple of (y_true, y_pred) as binary lists [0, 1]
    """
    # Convert correctness to binary predictions (1=correct, 0=incorrect)
    y_pred = list(map(lambda x: 1 if x else 0, correctness.tolist()))
    
    # Convert type strings to binary labels based on scope
    if scope is None or scope == 'I':
        # Inlier scope: 1 if type is 'I', else 0
        y_true = list(map(lambda x: 1 if x == 'I' else 0, instance_type.tolist()))
    elif scope == 'O':
        # Outlier scope: 1 if type is 'O', else 0
        y_true = list(map(lambda x: 1 if x == 'O' else 0, instance_type.tolist()))
    else:
        y_true = []
    
    return y_true, y_pred


def compute_auc_score(actual: list, predicted: list) -> float:
    """
    Calculate Area Under the ROC Curve (AUROC/AUC).
    
    AUROC measures the model's ability to distinguish between positive and negative
    classes across all possible classification thresholds.
    
    Formula:
        AUC = ∫ TPR(FPR) dFPR
    
    Where:
    - TPR (True Positive Rate): fraction of positives correctly identified
    - FPR (False Positive Rate): fraction of negatives incorrectly classified as positive
    
    Args:
        actual: Ground truth binary labels (0 or 1)
        predicted: Predicted anomaly scores (higher = more anomalous)
        
    Returns:
        AUC score in [0, 1]:
        - 1.0 = perfect classifier
        - 0.5 = random classifier (or undefined)
        - 0.0 = completely wrong classifier
    """
    try:
        return roc_auc_score(actual, predicted)
    except ValueError:
        # Return 0.5 if only one class present (undefined AUC)
        return 0.5


def compute_average_precision(actual: list, predicted: list,
                             sorted_indices: list) -> float:
    """
    Calculate Average Precision (AP) from precision-recall curve.
    
    AP integrates precision values across recall points where positives are ranked.
    This metric is particularly useful for imbalanced datasets where minority class
    (outliers) are the focus.
    
    Formula:
        AP = Σ (Recall_i - Recall_{i-1}) × Precision_i
    
    Where the sum is over all ranked instances from highest to lowest score.
    
    Args:
        actual: Ground truth binary labels (0 or 1)
        predicted: Predicted anomaly scores (higher = more anomalous)
        sorted_indices: Pre-computed indices sorted by prediction scores (descending)
        
    Returns:
        Average Precision in [0, 1]:
        - 1.0 = all outliers ranked above all inliers
        - 0.5 ≈ random ranking
        - 0.0 = all outliers ranked below all inliers
    """
    tp = 0  # True positives (outliers correctly ranked high)
    fp = 0  # False positives (inliers incorrectly ranked high)
    precision_values = []
    recall_values = []
    
    # Iterate through ranked instances from highest to lowest score
    for i in range(len(sorted_indices)):
        if actual[sorted_indices[i]] == 1:
            tp += 1
        else:
            fp += 1
        
        # Compute precision and recall at each threshold
        precision = tp / (tp + fp)
        total_positives = sum(actual)
        recall = tp / total_positives if total_positives > 0 else 0
        
        precision_values.append(precision)
        recall_values.append(recall)
    
    # Calculate area under precision-recall curve using trapezoid rule
    ap = 0.0
    for i in range(1, len(sorted_indices)):
        # Area under each step: (recall_change) × (precision at higher recall)
        ap += (recall_values[i] - recall_values[i - 1]) * precision_values[i]
    
    return ap


def compute_precision_at_k(actual: list, predicted: list, n: int) -> float:
    """
    Calculate Precision@n: fraction of top-n predictions that are correct.
    
    Also known as R-Precision when n equals the number of positive instances.
    This metric focuses on the quality of top-ranked predictions.
    
    Formula:
        P@n = (True Positives in top-n) / n
    
    For outlier detection, typically n = number of outliers in dataset (R-Precision).
    This ensures we evaluate only on the number of outliers to detect.
    
    Args:
        actual: Ground truth binary labels (0 or 1)
        predicted: Predicted anomaly scores (higher = more anomalous)
        n: Number of top-ranked instances to evaluate
        
    Returns:
        Precision@n in [0, 1]:
        - 1.0 = all top-n predictions are outliers
        - 0.5 = half of top-n predictions are outliers (random)
        - 0.0 = no top-n predictions are outliers
    """
    if n <= 0 or n > len(predicted):
        return 0.0
    
    # Sort indices by prediction scores (descending - highest scores first)
    sorted_indices = sorted(range(len(predicted)),
                           key=lambda i: predicted[i],
                           reverse=True)
    
    tp = 0  # Count of true outliers in top-n
    
    # Count true positives in top-n predictions
    for i in range(min(n, len(sorted_indices))):
        if actual[sorted_indices[i]] == 1:
            tp += 1
    
    return tp / n if n > 0 else 0.0


def compute_max_f1_score(y_true: list, y_pred: list) -> float:
    """
    Find the maximum F1-score by exploring different classification thresholds.
    
    Useful for continuous anomaly scores where the optimal threshold is unknown.
    Tests multiple thresholds and returns the best F1-score achieved.
    
    Formula:
        F1 = 2 × (Precision × Recall) / (Precision + Recall)
    
    The function tests thresholds: [0.1, 0.2, 0.3, ..., 0.9]
    
    Args:
        y_true: Ground truth binary labels (0 or 1)
        y_pred: Predicted anomaly scores in [0, 1]
        
    Returns:
        Maximum F1-score across all tested thresholds in [0, 1]
    """
    thresholds = np.arange(0.1, 1.0, 0.1)
    f1_scores = []
    
    # Test each threshold
    for threshold in thresholds:
        # Convert continuous scores to binary predictions using threshold
        y_pred_binary = [1 if score > threshold else 0 for score in y_pred]
        
        # Calculate F1-score for this threshold
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        f1_scores.append(f1)
    
    # Return best F1 (or 0 if no valid scores found)
    return np.max(f1_scores) if f1_scores else 0.0


# =============================================================================
# SECTION 3: Adjusted Metrics (Normalized by Class Imbalance)
# =============================================================================

def adjust_average_precision(ap: float, num_outliers: int,
                            num_instances: int) -> float:
    """
    Adjust Average Precision by random baseline for imbalanced datasets.
    
    Removes the effect of class imbalance on AP performance. The adjustment uses
    a random classifier baseline, which in imbalanced settings would achieve
    AP = minority_fraction.
    
    Normalization formula:
        Adjusted AP = (AP - baseline) / (1 - baseline)
    
    Where:
        baseline = num_outliers / num_instances (random classifier performance)
    
    Examples:
    - If outliers are 5% of data: baseline = 0.05
    - If AP = 0.40: Adjusted AP = (0.40 - 0.05) / (1 - 0.05) = 0.368
    - If AP = baseline: Adjusted AP = 0 (random classifier)
    - If AP = 1.0: Adjusted AP = 1.0 (perfect classifier)
    
    Args:
        ap: Original Average Precision score in [0, 1]
        num_outliers: Number of outlier instances in dataset
        num_instances: Total number of instances
        
    Returns:
        Adjusted AP (typically in [0, 1] but can be negative if AP < baseline)
    """
    # Calculate random classifier baseline
    baseline = num_outliers / num_instances if num_instances > 0 else 0
    
    # Calculate denominator (1 - baseline)
    denominator = 1 - baseline
    
    # Handle edge case where all instances are outliers
    if denominator <= 0:
        return 0.0
    
    # Apply normalization
    return (ap - baseline) / denominator


def adjust_r_precision(r_precision: float, num_outliers: int,
                      num_instances: int) -> float:
    """
    Adjust Precision@n (R-Precision) by random baseline for imbalanced datasets.
    
    Accounts for the fact that random guessing would achieve
    (num_outliers / num_instances) accuracy in imbalanced settings.
    
    This adjustment removes the bias introduced by class imbalance, allowing
    fair comparison across datasets with different outlier percentages.
    
    Normalization formula:
        Adjusted P@n = (P@n - baseline) / (1 - baseline)
    
    Where:
        baseline = num_outliers / num_instances
    
    Args:
        r_precision: Original Precision@n (P@R where R=num_outliers)
        num_outliers: Number of outlier instances in dataset
        num_instances: Total number of instances
        
    Returns:
        Adjusted P@n (typically in [0, 1] but can be negative if P@n < baseline)
    """
    # Calculate random classifier baseline
    baseline = num_outliers / num_instances if num_instances > 0 else 0
    
    # Calculate denominator (1 - baseline)
    denominator = 1 - baseline
    
    # Handle edge case where all instances are outliers
    if denominator <= 0:
        return 0.0
    
    # Apply normalization
    return (r_precision - baseline) / denominator


def adjust_maximum_f1(max_f1: float, num_outliers: int,
                     num_instances: int) -> float:
    """
    Adjust Maximum F1-score by random baseline for imbalanced datasets.
    
    Removes the effect of class imbalance on F1 performance, allowing fair
    comparison of algorithms across datasets with different outlier ratios.
    
    Normalization formula:
        Adjusted MaxF1 = (MaxF1 - baseline) / (1 - baseline)
    
    Where:
        baseline = num_outliers / num_instances
    
    Args:
        max_f1: Original Maximum F1-score
        num_outliers: Number of outlier instances in dataset
        num_instances: Total number of instances
        
    Returns:
        Adjusted Max F1 (typically in [0, 1] but can be negative if MaxF1 < baseline)
    """
    # Calculate random classifier baseline
    baseline = num_outliers / num_instances if num_instances > 0 else 0
    
    # Calculate denominator (1 - baseline)
    denominator = 1 - baseline
    
    # Handle edge case where all instances are outliers
    if denominator <= 0:
        return 0.0
    
    # Apply normalization
    return (max_f1 - baseline) / denominator


# =============================================================================
# SECTION 4: Aggregation & Analysis Functions
# =============================================================================

def calculate_adjusted_metrics(results_df: pd.DataFrame, dataset: str,
                              conversion_method: str) -> List[list]:
    """
    Calculate adjusted ranking-based metrics for a specific dataset.
    
    Orchestrates the computation of all ranking-based metrics (AUC, P@n, AP, Max-F1),
    adjusted for class imbalance. Processes all algorithms and parameters tested
    on the dataset and aggregates results by algorithm.
    
    Processing steps:
    1. Filter results for target dataset
    2. Iterate over all algorithms tested on dataset
    3. For each algorithm, iterate over all parameter values
    4. Compute all 4 metrics for each parameter combination
    5. Aggregate by algorithm (mean across parameters)
    6. Return one row per algorithm
    
    Args:
        results_df: Detailed execution results DataFrame with columns:
                    [algorithm, parameter, point, index, correct, dataset, type, score, ranking]
        dataset: Name of dataset to filter
        conversion_method: Conversion method name (BIN, BINDOWN, EXC, etc.)
        
    Returns:
        List of tuples: each tuple = (algorithm, dataset, method, AUC, P@n, AP, Max-F1)
    """
    result_metrics_auc = []
    result_metrics_pn = []
    result_metrics_ap = []
    result_metrics_f1 = []
    result_algorithms = []
    
    # Filter to current dataset
    df_dataset = results_df.query('dataset == @dataset')
    if len(df_dataset) == 0:
        return []
    
    # Iterate over algorithms tested on this dataset
    for algorithm in df_dataset['algorithm'].unique():
        df_algorithm = df_dataset.query('algorithm == @algorithm')
        
        tmp_auc = []
        tmp_pn = []
        tmp_ap = []
        tmp_f1 = []
        
        # Iterate over parameter values used for this algorithm
        for parameter in df_algorithm['parameter'].unique():
            df_parameter = df_algorithm.query('parameter == @parameter')
            
            # Convert type column to binary (1 if outlier, 0 if inlier)
            Y = list(map(lambda x: 1 if x == 'O' else 0,
                        df_parameter['type'].tolist()))
            
            # Skip if no outliers present
            if len(Y) == 0 or sum(Y) == 0:
                continue
            
            # Extract anomaly scores for this parameter
            scores = df_parameter['score'].tolist()
            n_outliers = sum(Y)
            n_instances = len(df_parameter)
            
            # ============================================================
            # Compute AUC (Area Under ROC Curve)
            # ============================================================
            auc = compute_auc_score(Y, scores)
            tmp_auc.append(auc)
            
            # ============================================================
            # Compute Precision@n (R-Precision) and adjust for imbalance
            # ============================================================
            r_precision = compute_precision_at_k(Y, scores, n_outliers)
            adjusted_r_prec = adjust_r_precision(r_precision, n_outliers, n_instances)
            tmp_pn.append(adjusted_r_prec)
            
            # ============================================================
            # Compute Average Precision and adjust for imbalance
            # ============================================================
            sorted_indices = sorted(range(len(scores)),
                                   key=lambda i: scores[i],
                                   reverse=True)
            ap = compute_average_precision(Y, scores, sorted_indices)
            adjusted_ap = adjust_average_precision(ap, n_outliers, n_instances)
            tmp_ap.append(adjusted_ap)
            
            # ============================================================
            # Compute Max-F1 and adjust for imbalance
            # ============================================================
            max_f1 = compute_max_f1_score(Y, scores)
            adjusted_max_f1 = adjust_maximum_f1(max_f1, n_outliers, n_instances)
            tmp_f1.append(adjusted_max_f1)
        
        # Filter NaN values and aggregate by algorithm (compute mean)
        tmp_auc = [x for x in tmp_auc if not np.isnan(x)]
        tmp_pn = [x for x in tmp_pn if not np.isnan(x)]
        tmp_ap = [x for x in tmp_ap if not np.isnan(x)]
        tmp_f1 = [x for x in tmp_f1 if not np.isnan(x)]
        
        # Only include algorithm if we have valid metrics for all types
        if tmp_auc and tmp_pn and tmp_ap and tmp_f1:
            # Clean dataset name by removing version suffixes (_v01, _v02, etc.)
            clean_dataset = re.sub('_v[0-9]+', '', dataset)
            
            result_algorithms.append([
                algorithm,
                clean_dataset,
                conversion_method,
                round(np.mean(tmp_auc), 4),
                round(np.mean(tmp_pn), 4),
                round(np.mean(tmp_ap), 4),
                round(np.mean(tmp_f1), 4)
            ])
    
    return result_algorithms


def process_metrics_batch(results_dir: str, output_path: str,
                         group_mapping: Dict[str, str]) -> pd.DataFrame:
    """
    Calculate metrics for all converted datasets across all conversion methods.
    
    Main processing pipeline that:
    1. Loads results for each conversion method
    2. Calculates metrics for each dataset-method combination
    3. Aggregates results by dataset and method
    4. Exports final metrics table to CSV
    
    Args:
        results_dir: Base directory containing detail_execution CSV files
        output_path: Path for output metrics CSV (including filename)
        group_mapping: Dict mapping conversion method names to results file paths
                      Example: {'BIN': 'path/to/BIN_detail_execution.csv', ...}
        
    Returns:
        DataFrame with aggregated metrics (columns: dataset, method, AUC, P@n, AP, Max-F1)
    """
    all_metrics = []
    
    # Process each conversion method
    for method, file_path in group_mapping.items():
        if not os.path.exists(file_path):
            print(f"⚠ File not found: {file_path}")
            continue
        
        print(f'\nProcessing: {method}')
        
        # Load detailed execution results for this method
        df_results = pd.read_csv(file_path, sep=';')
        
        # Get unique datasets in results
        datasets = df_results['dataset'].unique().tolist()
        
        # Process each dataset with progress bar
        with tqdm(total=len(datasets), desc="Calculating metrics",
                 unit="it", leave=True, colour="green") as pbar:
            for dataset in datasets:
                # Skip excluded datasets
                if dataset in EXCLUDED_DATASETS:
                    pbar.update(1)
                    continue
                
                # Calculate metrics for this dataset-method combination
                metrics = calculate_adjusted_metrics(df_results, dataset, method)
                all_metrics.extend(metrics)
                pbar.update(1)
    
    # Create DataFrame from collected metrics
    df_metrics = pd.DataFrame(all_metrics, columns=METRICS_COLUMNS)
    
    # Aggregate by dataset and method (compute mean and std for each metric)
    df_metrics = df_metrics.groupby(['dataset', 'method'], as_index=False).agg(
        AGGREGATION_FUNCTIONS
    ).sort_values(by=['method', 'dataset']).reset_index(drop=True)
    
    # Save aggregated metrics to CSV
    df_metrics.to_csv(output_path, sep=';', index=False)
    print(f"\n✓ Metrics saved to: {output_path}")
    
    return df_metrics


def describe_datasets(directory_list: Dict[str, str]) -> pd.DataFrame:
    """
    Generate statistical summary of all datasets in specified directories.
    
    Scans directories for CSV files and extracts:
    - Dataset name (filename)
    - Number of instances (rows)
    - Number of outliers (rows with outlier='yes')
    - Number of features (columns - 1 for label)
    - Dataset group/category
    
    Args:
        directory_list: Dict mapping group names to directory paths
                       Example: {'BIN': 'path/to/BIN/', 'EXC': 'path/to/EXC/', ...}
        
    Returns:
        DataFrame with columns: [Dataset, # Instances, # Outliers, # Features, Group]
    """
    dataset_list = []
    
    # Iterate over dataset groups
    for group_name, directory_path in directory_list.items():
        # Skip if directory doesn't exist
        if not os.path.exists(directory_path):
            print(f"⚠ Directory not found: {directory_path}")
            continue
        
        # List all files in directory
        datasets = os.listdir(directory_path)
        
        # Process each file
        for dataset in datasets:
            file_path = os.path.join(directory_path, dataset)
            
            # Only process CSV files that are regular files (not directories)
            if not os.path.isfile(file_path) or not dataset.endswith('.csv'):
                continue
            
            # Load dataset
            df = pd.read_csv(file_path, sep=';')
            
            # Count outlier instances
            n_outliers = len(df.query('outlier == "yes"'))
            
            # Extract statistics
            dataset_list.append([
                dataset,
                len(df),  # Number of instances (total rows)
                n_outliers,  # Number of outliers
                len(df.columns) - 1,  # Number of features (exclude label column)
                group_name  # Dataset group/category
            ])
    
    # Create DataFrame with statistics
    return pd.DataFrame(dataset_list,
                       columns=['Dataset', '# Instances', '# Outliers',
                               '# Features', 'Group'])

In [7]:
# =============================================================================
# SECTION 5: Main Execution Pipeline
# =============================================================================

def main():
    """
    Main execution pipeline for metrics calculation.
    
    Orchestrates complete workflow:
    1. Configure input/output paths for all conversion methods
    2. Load and process detailed execution results
    3. Calculate adjusted ranking-based metrics
    4. Generate dataset statistics
    5. Export results to CSV files
    
    Expected directory structure:
        results/
        ├── binary/
        │   ├── BIN/
        │   │   └── BIN_detail_execution.csv
        │   └── BINDOWN/
        │       └── BINDOWN_detail_execution.csv
        └── non_binary/
            ├── EXC/
            │   └── EXC_detail_execution.csv
            ├── EXCDOWN/
            │   └── EXCDOWN_detail_execution.csv
            ├── GRO/
            │   └── GRO_detail_execution.csv
            └── GRODOWN/
                └── GRODOWN_detail_execution.csv
    """
    print("\n" + "="*70)
    print("Metrics Calculation Pipeline")
    print("="*70)
    
    # ========================================================================
    # Configuration: Define input and output paths
    # ========================================================================
    base_dir = 'conversion_methods'
    datasets_dir = os.path.join(r'..\..\datasets', base_dir)
    results_dir = os.path.join(r'..\..\results', base_dir)
    
    # Create output directories if they don't exist
    os.makedirs(results_dir, exist_ok=True)
    
    # ========================================================================
    # Configure results files for binary conversion methods
    # ========================================================================
    binary_results = {
        'BIN': os.path.join(results_dir, 'binary', 'converted', 'BIN',
                           'BIN_detail_execution.csv'),
        'BINDOWN': os.path.join(results_dir, 'binary', 'converted', 'BINDOWN',
                               'BINDOWN_detail_execution.csv'),
    }
    
    # ========================================================================
    # Configure results files for non-binary conversion methods
    # ========================================================================
    non_binary_results = {
        'EXC': os.path.join(results_dir, 'converted', 'non_binary', 'EXC',
                           'EXC_detail_execution.csv'),
        'EXCDOWN': os.path.join(results_dir, 'converted', 'non_binary', 'EXCDOWN',
                               'EXCDOWN_detail_execution.csv'),
        'GRO': os.path.join(results_dir, 'converted', 'non_binary', 'GRO',
                           'GRO_detail_execution.csv'),
        'GRODOWN': os.path.join(results_dir, 'converted', 'non_binary', 'GRODOWN',
                               'GRODOWN_detail_execution.csv'),
    }
    
    # Combine all results mapping
    all_results = {**binary_results, **non_binary_results}
    
    # ========================================================================
    # Step 1: Process and calculate metrics
    # ========================================================================
    print("\n[1/2] Processing metrics from detailed execution results...")
    df_metrics = process_metrics_batch(
        results_dir,
        os.path.join(results_dir, 'metrics.csv'),
        all_results
    )
    
    # ========================================================================
    # Step 2: Generate dataset statistics
    # ========================================================================
    print("\n[2/2] Generating dataset statistics...")
    dataset_dirs = {
        'BIN': os.path.join(datasets_dir, 'binary', 'converted', 'BIN'),
        'EXC': os.path.join(datasets_dir, 'non_binary', 'converted', 'EXC'),
    }
    df_datasets = describe_datasets(dataset_dirs)
    print(f"✓ Dataset statistics: {len(df_datasets)} datasets")
    print(f"\nDataset Summary:\n{df_datasets}")
    
    # ========================================================================
    # Pipeline Summary
    # ========================================================================
    print("\n" + "="*70)
    print("Pipeline Complete")
    print("="*70)
    print(f"\n✓ Output saved to: {os.path.join(results_dir, 'metrics.csv')}")
    print(f"✓ Processed conversion methods: {', '.join(all_results.keys())}")
    print(f"✓ Total datasets analyzed: {len(df_datasets)}")
    print()


if __name__ == "__main__":
    main()


Metrics Calculation Pipeline

[1/2] Processing metrics from detailed execution results...

Processing: BIN


Calculating metrics: 100%|██████████| 14/14 [02:23<00:00, 10.24s/it]


⚠ File not found: ..\..\results\conversion_methods\binary\converted\BINDOWN\BINDOWN_detail_execution.csv
⚠ File not found: ..\..\results\conversion_methods\converted\non_binary\EXC\EXC_detail_execution.csv
⚠ File not found: ..\..\results\conversion_methods\converted\non_binary\EXCDOWN\EXCDOWN_detail_execution.csv
⚠ File not found: ..\..\results\conversion_methods\converted\non_binary\GRO\GRO_detail_execution.csv
⚠ File not found: ..\..\results\conversion_methods\converted\non_binary\GRODOWN\GRODOWN_detail_execution.csv

✓ Metrics saved to: ..\..\results\conversion_methods\metrics.csv

[2/2] Generating dataset statistics...
✓ Dataset statistics: 35 datasets

Dataset Summary:
                     Dataset  # Instances  # Outliers  # Features Group
0                 autism.csv          604         178          20   BIN
1               autistic.csv          246         121          20   BIN
2               banknote.csv         1348         610           4   BIN
3      blood_transfusion.csv 